# Lab 0 Assignment: Verify Your Environment With Code

Use this notebook after you finish `03_environment_check.ipynb`.

This assignment asks you to run and complete a few short code tasks so you can prove your local setup is working.

You will:
- locate the repo root in code
- load `.env` in code
- contact the configured Ollama endpoint in code
- confirm the default model is available
- ask the configured model one question
- build a short environment summary in Python
- observe what your endpoint returns and report it clearly

In [1]:
import json
from pathlib import Path

import requests
from dotenv import dotenv_values
from openai import OpenAI

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.env.example').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not find the repo root.')

repo_root = find_repo_root(Path.cwd().resolve())
print('Detected repo root:', repo_root)

Detected repo root: /home/frank/projects/agentic-AI4-forensics


## Task 1

Run the cell below and confirm that the three required repo paths exist.

Expected result:
- `.env` exists
- `requirements.txt` exists
- `src/` exists

In [2]:
# Run this cell and confirm that each required path exists.
env_path = repo_root / '.env'
requirements_path = repo_root / 'requirements.txt'
src_path = repo_root / 'src'

print('.env exists:', env_path.exists())
print('requirements.txt exists:', requirements_path.exists())
print('src exists:', src_path.exists())

.env exists: True
requirements.txt exists: True
src exists: True


## Task 2

Load `.env` and print the configured model and endpoint.

In [3]:
config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

print('MODEL:', model)
print('OLLAMA_BASE_URL:', ollama_base_url)

if not model or not ollama_base_url:
    raise ValueError('MODEL or OLLAMA_BASE_URL is missing from .env')

MODEL: qwen3:8b
OLLAMA_BASE_URL: http://localhost:11434/v1


## Task 3

Build the `/api/tags` URL from `OLLAMA_BASE_URL`, send the request, and list the available models.

In [4]:
api_root = ollama_base_url.rstrip('/')
if api_root.endswith('/v1'):
    tags_url = api_root[:-3] + '/api/tags'
else:
    tags_url = api_root + '/api/tags'

response = requests.get(tags_url, timeout=10)
response.raise_for_status()
available_models = [item.get('name') for item in response.json().get('models', []) if item.get('name')]

print('Tags URL:', tags_url)
print('Available models:')
for model_name in available_models:
    print('-', model_name)

Tags URL: http://localhost:11434/api/tags
Available models:
- gemma4:e4b
- huihui_ai/qwen3.5-abliterated:latest
- qwen3.5:2b
- qwen3.5:9b
- qwen3.5:0.8b
- qwen3.5:35b-a3b
- qwen3:8b
- qwen3.5:27b
- llama3.3:70b


## Task 4

Check whether the default model from `.env` is available at the configured endpoint.

In [5]:
model_available = model in available_models

print('Default model available:', model_available)

if not model_available:
    raise ValueError(f'The model {model!r} is not available at the configured endpoint.')

Default model available: True


## Task 5

Ask the configured model one question.

Replace the example text in `student_question` with one question of your own, then run the cell.

In [ ]:
# Replace the example text below with one question of your own.
student_question = 'What is one good habit for organizing digital evidence files?'

client = OpenAI(base_url=ollama_base_url, api_key='ollama')
response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': student_question}],
)

model_answer = response.choices[0].message.content.strip()

print('Question:', student_question)
print('\nModel answer:')
print(model_answer)

## Task 6

Run the cell below and review the environment summary as formatted JSON.

For `repo_root_name`, use only the folder name, not the full absolute path.

In [6]:
# Run this cell and review the summary values before you move to Task 7.
environment_summary = {
    'repo_root_name': repo_root.name,
    'default_model': model,
    'ollama_base_url': ollama_base_url,
    'available_model_count': len(available_models),
    'default_model_available': model_available,
}

print(json.dumps(environment_summary, indent=2))

{
  "repo_root_name": "agentic-AI4-forensics",
  "default_model": "qwen3:8b",
  "ollama_base_url": "http://localhost:11434/v1",
  "available_model_count": 9,
  "default_model_available": true
}


## Task 7: Observation Report

This is a short write-up task based on the results you already printed in Tasks 3 through 6.

You do not need to contact the endpoint again. Just look at your earlier outputs and fill in the report below.

Type your answers by replacing the five empty strings in the `setup_report` code cell below.

Include:
- how many models were returned in the printed model list from Task 3
- whether the default model from `.env` was available
- one other model name you noticed in the list
- whether your environment is ready for `lab0_2_model_warmup`
- a short 1-2 sentence explanation of your answer

In [ ]:
# TODO: use your outputs from Tasks 3 through 6 to complete this short report.
# Replace all five empty strings with your own answers.
# Use plain-language strings such as '9', 'available', or 'yes'.
setup_report = {
    # Enter how many models were returned in the printed model list from Task 3.
    'observed_model_count': '',
    # Enter whether the default model from .env was available.
    'default_model_status': '',
    # Enter one additional model you noticed in the printed list.
    'one_other_model_you_noticed': '',
    # State whether your environment is ready for the next lab: yes or no.
    'ready_for_lab0_2_model_warmup': '',
    # Explain your readiness answer in 1-2 sentences.
    'readiness_explanation': '',
}

required_text_fields = [
    'observed_model_count',
    'default_model_status',
    'one_other_model_you_noticed',
    'ready_for_lab0_2_model_warmup',
    'readiness_explanation',
]

missing = [key for key in required_text_fields if not str(setup_report[key]).strip()]

print(json.dumps(setup_report, indent=2))

if missing:
    print(f"\nPlease complete these fields before submitting: {', '.join(missing)}")
else:
    print("\nObservation report complete.")

## Submission

Save the notebook with:
- the completed code cells
- the printed model list
- the printed question and model answer from Task 5
- the final `environment_summary` output
- the completed `setup_report` observation report